# Exercise 3: Extending an existing analyser

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Endangered Languages and Ethics · version 0.1.0*

> Time a small extension to an existing FST. Include linguistic review, ambiguity tests and maintenance in the effort estimate, not only dictionary entry counts.

**Setup.** `pip install requests huggingface_hub fasttext-wheel transformers torch sentencepiece "turkicnlp[hfst]==0.5.0"`; each notebook uses only part of this, and Exercise 1 also calls `git`. Data come from commit- or tag-pinned GitHub files and are hash-checked: the UD 2.17 TueCL treebanks, the KazNERD and KyrgyzNER releases and their README files, and the TIL corpus README. The audit in Exercise 1 also reads the book's UD 2.17 inventory (`tables/tbl-ud217-inventory/data.csv`), the NLLB-200 tokeniser (`NLLB_PATH`) and GlotLID (`GLOTLID_PATH`), and queries live services, so its live columns are dated observations, not pinned results. Run the notebooks from the `exercises/` folder: the first code cell imports the chapter's `observation` and `hub_count` helpers from `../code/remaining/resources.py`. Downloads are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`).

The extension adds place names that TurkicNLP's Apertium Kazakh analyser does not know, with six case forms generated by explicit harmony and assimilation rules, and unions it with the analyser (HFST). The rules are first tested on place names the analyser already knows; regression tests check that no existing reading changes and that wrong-harmony forms are rejected. Candidates come from the KazNERD training split; coverage is measured on the test split.

In [ ]:
import collections, csv, datetime, difflib, hashlib, json, os, re, statistics, subprocess, sys, time
from pathlib import Path
import requests
sys.path.insert(0, str(Path("../code/remaining").resolve()))
from resources import observation, hub_count

CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
KAZNERD = "https://raw.githubusercontent.com/IS2AI/KazNERD/bd4333d0f5952b9fafb2ef2ac2fefa0ad3c0333f/"
KYRGYZNER = "https://raw.githubusercontent.com/Akyl-AI/KyrgyzNER/b76b9c9c9c4f5d681c25308cc0d8dd9f0bd6569c/"
TIL = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/7d988cc47dba889d02262d893bf9a46e43a6bb36/"
SIB = "https://raw.githubusercontent.com/dadelani/sib-200/efad32e0fc601cfbfdf5ca80672f29697f6f401f/data/annotated/"
UD = "https://raw.githubusercontent.com/UniversalDependencies/{}/r2.17/{}"
SHA256 = {
    KAZNERD + "README.md": "2d56859aa35eadafb8bc560b375d5128627e3bab3517bc74377c7818905e3614",
    KAZNERD + "KazNERD/IOB2_train.txt": "e5a80f6d4bbff499a4bf7cbf96a169feb761411151fb45abebeaae06995ff647",
    KAZNERD + "KazNERD/IOB2_test.txt": "359a1f61f6a21280c6c47b4fde9a5adcd07411d62015aa6e32d940c51f163b7a",
    KYRGYZNER + "README.md": "c5beec0dd94156f5cdae4c5f7ffafeb23704f6fd1024ed4cb505cc8ef498a46c",
    TIL + "til_corpus/README.md": "da88f81cc7a23fb46425dcf236f7c89ad0f63663d2141d8fb7165ea8015308d8",
    UD.format("UD_Turkish-TueCL", "tr_tuecl-ud-test.conllu"): "2508f47e57b9b6a1d7241d0ab4a918d84c5cbb0455bbd961ac5231e31a1d311e",
    UD.format("UD_Azerbaijani-TueCL", "az_tuecl-ud-test.conllu"): "8b315f8b065f094e38482465706ba44a9caf1b082fc8bbc94908f6fdeee8c3b5",
    UD.format("UD_Uzbek-TueCL", "uz_tuecl-ud-test.conllu"): "23257711196df0bbfaf5b25397d7c6102fbc8800ed83b15eb77342929ee0e469",
    UD.format("UD_Kyrgyz-TueCL", "ky_tuecl-ud-test.conllu"): "08224be6b74346f582234f54527412dd6265c8d8b42e7d6ef463aade85bdcd9b",
}
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    h = hashlib.sha256(path.read_bytes()).hexdigest(); FETCHED[url] = h
    if url in SHA256 and h != SHA256[url]:
        raise ValueError(f"{url} differs from the pinned file")
    return path

def read_conll(path):
    """KazNERD-style column file -> [(words, tags)]."""
    sents, words, tags = [], [], []
    for line in Path(path).read_text(encoding="utf-8").splitlines() + [""]:
        if not line.strip():
            if words: sents.append((words, tags))
            words, tags = [], []
        elif not line.startswith("# "):
            c = line.split("\t") if "\t" in line else line.split()
            words.append(c[0]); tags.append(c[-1].strip())
    return sents

def show(obs):
    """One audit cell: the value, or the reason why there is none. Zero, not found, error and not checked stay distinct."""
    return {"observed": lambda o: str(o["value"]), "not_found": lambda o: "not found", "error": lambda o: "ERROR",
            "not_checked": lambda o: "not checked"}[obs["status"]](obs)
print("helpers loaded from code/remaining/resources.py:", observation.__name__, hub_count.__name__)

In [ ]:
import hfst
from turkicnlp.resources.registry import ModelRegistry
T = {}
def tic(step):
    T[step] = time.perf_counter()
def toc(step):
    T[step] = time.perf_counter() - T[step]
tic("load analyser")
path = ModelRegistry.get_model_path("kaz", "morph", "apertium", script="Cyrl") / "kaz.automorf.hfst"
base = hfst.HfstInputStream(str(path)).read()
toc("load analyser")
def readings(fst, w):
    return sorted({r for r, wt in fst.lookup(w)})
train = read_conll(fetch(KAZNERD + "KazNERD/IOB2_train.txt")); test = read_conll(fetch(KAZNERD + "KazNERD/IOB2_test.txt"))
print(f"analyser: TurkicNLP Apertium kaz (sha256 {hashlib.sha256(path.read_bytes()).hexdigest()[:12]})")
# 1. Candidate stems: single-word place mentions in the TRAINING split that the analyser does not know.
tic("find candidates")
known = {}
def ok(w):
    if w not in known: known[w] = bool(readings(base, w))
    return known[w]
single = collections.Counter(ws[i] for ws, ts in train for i, t in enumerate(ts)
                             if t in ("B-GPE", "B-LOCATION") and (i + 1 == len(ts) or not ts[i + 1].startswith("I-")) and ws[i].isalpha())
cands = [w for w, n in single.most_common() if n >= 3 and w[0].isupper() and not ok(w)]
inflected = [w for w in cands if any(w != s and w.startswith(s) for s in cands)]      # e.g. a locative form of another candidate
cands = [w for w in cands if w not in inflected]
toc("find candidates")
# 2. Case endings by rule (back/front harmony, stem-final sound). Written for this exercise; not reviewed by a speaker.
BACK, FRONT = set("аоұыяу"), set("әөүіеи")
VOICELESS = set("пткқсшфхцчщбвгд")      # word-final б в г д are pronounced voiceless in loanwords
NASAL, SONOR, VOICED = set("мнң"), set("рйуил"), set("жз")
VOWEL = set("аәеоөұүыіэюя")
def harmony(stem):
    for ch in reversed(stem.lower()):
        if ch in BACK - {"у"}: return 0
        if ch in FRONT - {"и"}: return 1
    return 0
def endings(stem):
    last, h = stem[-1].lower(), harmony(stem)
    pick = lambda back, front: (back, front)[h]
    if last in VOWEL:            g, a, d, l, ab = "ның нің", "ны ні", "ға ге", "да де", "дан ден"
    elif last in NASAL:          g, a, d, l, ab = "ның нің", "ды ді", "ға ге", "да де", "нан нен"
    elif last in SONOR | VOICED: g, a, d, l, ab = "дың дің", "ды ді", "ға ге", "да де", "дан ден"
    else:                        g, a, d, l, ab = "тың тің", "ты ті", "қа ке", "та те", "тан тен"
    forms = {"nom": ""}
    for case, pair in (("gen", g), ("acc", a), ("dat", d), ("loc", l), ("abl", ab)):
        forms[case] = pick(*pair.split())
    return forms
def wrong(stem):
    """The same endings with the opposite harmony: forms a correct analyser should reject."""
    h = harmony(stem); flip = {0: 1, 1: 0}[h]
    pairs = {"ның": "нің", "дың": "дің", "тың": "тің", "ны": "ні", "ды": "ді", "ты": "ті", "ға": "ге", "қа": "ке",
             "да": "де", "та": "те", "дан": "ден", "нан": "нен", "тан": "тен"}
    pairs.update({v: k for k, v in list(pairs.items())})
    return {c: pairs[e] for c, e in endings(stem).items() if e}
# 3. Regression test of the rules on place names the analyser already knows: generated forms must get the same reading.
known_places = [w for w, n in single.most_common(400) if w[0].isupper() and any("<np><top><nom>" in r for r in readings(base, w))][:100]
agree = n_rule = 0; mismatches = []; wrong_kind = collections.Counter()
for s in known_places:
    for case, e in endings(s).items():
        n_rule += 1
        if f"{s}<np><top><{case}>" in readings(base, s + e): agree += 1
        else: mismatches.append(s + e)
    for case, e in wrong(s).items():
        rs = [r for r in readings(base, s + e) if r.startswith(s + "<np><top>")]
        wrong_kind["rejected" if not rs else "flagged <err_orth>" if all("<err_orth>" in r for r in rs) else "accepted"] += 1
print(f"rule check on {len(known_places)} known place names: {agree}/{n_rule} generated forms receive the expected reading; "
      f"mismatches e.g. {mismatches[:6]}")
print(f"the same names with wrong-harmony endings, as read by the analyser: {dict(wrong_kind)}")
# 4. Build the extension and union it with the analyser.
tic("compile extension")
pairs = {s + e: f"{s}<np><top><{case}>" for s in cands for case, e in endings(s).items()}
ext = hfst.fst(pairs)
toc("compile extension")
tic("union and convert")
merged = hfst.HfstTransducer(base); merged.convert(hfst.ImplementationType.TROPICAL_OPENFST_TYPE)
merged.disjunct(ext); merged.convert(hfst.ImplementationType.HFST_OLW_TYPE)
toc("union and convert")
print(f"extension: {len(cands)} unknown place names seen at least 3 times as single-word GPE/LOCATION mentions in training "
      f"({len(inflected)} more set aside as inflected forms of another candidate, e.g. {inflected[:3]}), {len(pairs):,} surface forms; e.g. {cands[:8]}")
# 5. Coverage on the TEST split, and regressions on words the analyser already handled.
tic("evaluate")
tokens = [w for ws, _ in test for w in ws if w.isalpha()]
ent = [w for ws, ts in test for w, t in zip(ws, ts) if w.isalpha() and t[2:] in ("GPE", "LOCATION")]
types = sorted(set(tokens))
before = {w: readings(base, w) for w in types}; after = {w: readings(merged, w) for w in types}
cov = lambda ws, R: sum(bool(R[w]) for w in ws) / len(ws)
changed = [w for w in types if before[w] and before[w] != after[w]]
toc("evaluate")
print(f"test tokens: coverage {cov(tokens, before):.2%} -> {cov(tokens, after):.2%}; GPE/LOCATION tokens {cov(ent, before):.2%} -> {cov(ent, after):.2%}")
print(f"previously analysed test word types whose readings changed: {len(changed)} of {sum(bool(v) for v in before.values()):,}, e.g. {changed[:3]}")
new_ok = [w for w in types if not before[w] and after[w]]
print(f"newly covered test word types: {len(new_ok)}, e.g. {new_ok[:10]}")
print("wrong-harmony forms of the new stems accepted by the extended analyser:",
      sum(bool(readings(merged, s + e)) for s in cands for e in wrong(s).values()), "of", sum(len(wrong(s)) for s in cands))
print("machine time (s):", {k: round(v, 2) for k, v in T.items()})

## Worked solution

Output of `experiments/ch17_exercises/run_ch17_exercises.py`:

```
analyser: TurkicNLP Apertium kaz (sha256 902af879f588)
rule check on 100 known place names: 584/600 generated forms receive the expected reading; mismatches e.g. ['Беларусьтың', 'Беларусьты', 'Беларусьқа', 'Беларусьта', 'Беларусьтан', 'Ирак']
the same names with wrong-harmony endings, as read by the analyser: {'flagged <err_orth>': 194, 'rejected': 306}
extension: 149 unknown place names seen at least 3 times as single-word GPE/LOCATION mentions in training (14 more set aside as inflected forms of another candidate, e.g. ['Қордайдағы', 'Шамалғаннан', 'Қаскелеңдегі']), 894 surface forms; e.g. ['Мұнайлы', 'Ухань', 'Қазалы', 'Шардара', 'Қордай', 'Қосшы', 'Қармақшы', 'Қиғаш']
test tokens: coverage 85.75% -> 85.88%; GPE/LOCATION tokens 85.20% -> 89.52%
previously analysed test word types whose readings changed: 0 of 16,470, e.g. []
newly covered test word types: 64, e.g. ['Іле', 'Анталия', 'Арқа', 'Арқаның', 'Астрахань', 'Бейбітшілік', 'Бейбітшілікті', 'Ботай', 'Бәйтерек', 'Зерделі']
wrong-harmony forms of the new stems accepted by the extended analyser: 0 of 745
machine time (s): {'load analyser': 0.02, 'find candidates': 0.06, 'compile extension': 6.02, 'union and convert': 0.93, 'evaluate': 1.37}
```

**Agreement with an analyser is a regression check.** The exercise's rules reproduce an expected analyser reading for 584 of 600 generated forms of known place names. This is agreement with the existing implementation, not independent linguistic validation. Inspect the mismatches before extending the rules.

Among 500 deliberately altered endings on known stems, 194 receive readings flagged `<err_orth>` and 306 are rejected. A nonempty reading therefore does not necessarily mean the analyser treats the spelling as standard.

The extension adds 894 forms from 149 candidate stems drawn from training data. Test GPE/LOCATION token coverage rises from 85.20% to 89.52%, while overall coverage rises from 85.75% to 85.88%. No previously analysed test type changes its readings, and none of 745 altered endings on new stems is accepted. These checks cover the sampled forms, not all possible regressions. Candidate stems and new readings still need speaker review.

Compilation takes about six seconds in the recorded run. Candidate selection, linguistic review and maintenance were not timed, so this is not an estimate of development effort.

## Your turn — extension

With a speaker, select appropriate possessive or plural forms for a second continuation class. Extend the regression tests and time work on 20 new stems, including review. Then propose the entries in the format of the `apertium-kaz` lexicon.